# 01 — Data acquisition and initial data-quality audit

This notebook examines the UCI *Default of Credit Card Clients* dataset (ID 350) before predictive modelling. We inspect the source, column definitions, missing values, duplicates, plausibility and outcome balance. The sample contains existing cardholders, not a representative set of all applicants.

## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

## 2. Configure pandas display

Show all columns and make numeric output easier to read. These settings do not change the underlying data.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 3. Set up project folders

Locate the repository root and create directories for raw and processed data, plots and exported summary tables.

In [ ]:
current_directory = Path.cwd()
if current_directory.name == "notebooks":
    project_root = current_directory.parent 
else:
    project_root = current_directory 

folders = [
    project_root / "data" / "raw",
    project_root / "data" / "processed",
    project_root / "outputs" / "figures",
    project_root / "outputs" / "tables"
]

for folder in folders:
    folder.mkdir(parents = True, exist_ok = True)

print(f"Project_root: {project_root}")

## 4. Retrieve and document the dataset

Load dataset 350 from the UCI repository and inspect variable definitions, metadata and available data components before interpreting encoded fields.

In [ ]:
credit_data = fetch_ucirepo(id = 350)
print(credit_data.variables)
print(credit_data.metadata["additional_info"]["variable_info"])
print(credit_data.data.keys())

## 5. Inspect the features, outcome and record IDs

Keep predictors, target and customer identifiers separate initially and verify their respective dimensions.

In [ ]:
feature = credit_data.data.features.copy()
target = credit_data.data.targets.copy()
customer_ids = credit_data.data.ids.copy()

print("Feature Shape:", feature.shape)
print("Target Shape:", target.shape)
print("ID Shape:", customer_ids.shape)

## 6. Assemble the working DataFrame

Join the fields into one dataset and preview records to check their alignment.

In [ ]:
df = pd.concat([customer_ids, feature, target], axis=1)

print("Combined Shape:", df.shape)
df.head(10)

## 7. Rename variables

Map generic source names such as `X1` to descriptive labels for credit limit, payment status, monthly bills and payments. Rename the binary target to `default_next_month`.

In [ ]:
rename_map = {
    "ID": "customer_id",
    "X1": "credit_limit",
    "X2": "sex",
    "X3": "education",
    "X4": "marriage",
    "X5": "age",
    "X6": "repayment_status_sep",
    "X7": "repayment_status_aug",
    "X8": "repayment_status_jul",
    "X9": "repayment_status_jun",
    "X10": "repayment_status_may",
    "X11": "repayment_status_apr",
    "X12": "bill_amount_sep",
    "X13": "bill_amount_aug",
    "X14": "bill_amount_jul",
    "X15": "bill_amount_jun",
    "X16": "bill_amount_may",
    "X17": "bill_amount_apr",
    "X18": "payment_amount_sep",
    "X19": "payment_amount_aug",
    "X20": "payment_amount_jul",
    "X21": "payment_amount_jun",
    "X22": "payment_amount_may",
    "X23": "payment_amount_apr"
}

target_column = target.columns[0]
rename_map[target_column] = "default_next_month"

df = df.rename(columns = rename_map)
df.columns.tolist()

## 8. Inspect dimensions, data types and descriptive statistics

Review the number of records and columns, sample rows, inferred data types and numerical summaries.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 9. Audit missing values

Count null values for every column, calculate each column's missing percentage and sort the results. Even if missingness is zero, undocumented category codes may still need attention.

In [ ]:
missing_table = df.isna().sum().to_frame(name = "missing_count")

missing_table["missing_percentage"] = (missing_table["missing_count"] / len(df) * 100)

missing_table = missing_table.sort_values("missing_percentage", ascending = False)

missing_table.head()

## 10. Check duplicate rows and customer IDs

Investigate repeated records and identifiers. Unique IDs will be important for interpreting later customer-level modelling and validation.

In [ ]:
duplicated_rows = df.duplicated().sum()
duplicated_ids = df["customer_id"].duplicated().sum()

print(f"Duplicated rows: {duplicated_rows}") 
print(f"Duplicated IDs: {duplicated_ids}") 

## 11. Examine categorical codes

Review the frequencies of sex, education, marriage and outcome values, including nulls. Unexpected demographic codes should be assessed and handled transparently rather than silently discarded.

In [ ]:
categorical_columns = [
    "sex",
    "education",
    "marriage",
    "default_next_month",
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna = False).sort_index())

## 12. Examine monthly repayment-status codes

Inspect observed minima and maxima for each of the six monthly repayment-status fields, then count the frequency of all recorded codes. Negative status codes need careful interpretation against the dataset documentation.

In [ ]:
repayment_columns = [
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",
]

repayment_range = df[repayment_columns].agg(["min", "max"]).T
repayment_range 

In [ ]:
repayment_code_counts = (
    df[repayment_columns]
    .stack()
    .value_counts()
    .sort_index()
)
repayment_code_counts

## 13. Identify billing and payment columns

Create lists of monthly bill amounts and payments to support subsequent data-validity checks.

In [ ]:
bill_columns = [
    "bill_amount_sep",
    "bill_amount_aug",
    "bill_amount_jul",
    "bill_amount_jun",
    "bill_amount_may",
    "bill_amount_apr"
]

payment_columns = [
    "payment_amount_sep",
    "payment_amount_aug",
    "payment_amount_jul",
    "payment_amount_jun",
    "payment_amount_may",
    "payment_amount_apr"
]

## 14. Check potentially invalid or unusual values

Count nonpositive credit limits, extreme ages, negative payments and negative bill balances. A negative bill balance can reflect a refund or credit and is not necessarily an error.

The `invalid_target_values` calculation uses `(~df['default_next_month'].isin([0, 1])).sum()` to count values outside the valid binary labels `{0, 1}`. The `~` operator negates the membership check so that only invalid target values are counted.

In [ ]:
validity_checks = pd.Series({
    "non_positive_credit_limits" :
    (df["credit_limit"] <= 0).sum(),

    "customers_under_18" :
    (df["age"] < 18).sum(),

    "customers_over_100" :
    (df["age"] > 100).sum(),

    "negative_payment_amounts" :
    (df[payment_columns] < 0).sum().sum(),

    "negative_bill_amounts" :
    (df[bill_columns] < 0).sum().sum(),

    "invalid_target_values" :
    (~df["default_next_month"].isin([0, 1])).sum()
})

validity_checks

## 15. Calculate outcome prevalence

Summarise the class counts and percentages for default versus non-default. This gives a baseline for thinking about class imbalance and later evaluation metrics.

In [ ]:
default_summary = (df["default_next_month"]
                  .value_counts()
                  .sort_index()
                  .rename_axis("default_status")
                  .to_frame("customers")
                 )

default_summary["default_percentage"] = (default_summary["customers"] / len(df) * 100)

default_summary

## 16. Plot the outcome distribution

Visualise default and non-default counts, label the bars and save the figure in `outputs/figures/` for reuse.

In [ ]:
default_counts = (
    df["default_next_month"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    ["No default", "Default"],
    default_counts.values,
    color=["#4C78A8", "#E45756"],
)

ax.set_title("Default Status in the Following Month")
ax.set_xlabel("Customer outcome")
ax.set_ylabel("Number of customers")

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()

figure_path = (
    project_root
    / "outputs"
    / "figures"
    / "default_distribution.png"
)

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

## 17. Save the local input for the next notebook

Export the renamed data to `data/raw/credit_default_uci.csv`. The next notebook reads this file. Customer-level raw files are excluded from version control.

In [ ]:
raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "credit_default_uci.csv"
)

df.to_csv(raw_data_path, index=False)

## 18. Export the audit tables

Save missingness, validity checks and outcome summaries under `outputs/tables/`. These are generated outputs and can be reproduced by rerunning this notebook.

In [ ]:
missing_table.to_csv(
    project_root / "outputs" / "tables" / "missing_values.csv"
)

validity_checks.to_csv(
    project_root / "outputs" / "tables" / "validity_checks.csv",
    header=["count"],
)

default_summary.to_csv(
    project_root / "outputs" / "tables" / "default_summary.csv"
)

## Initial data-quality assessment

The dataset contains 30,000 customer records, 23 predictor variables, one binary target variable, and one customer identifier. 
Each row represents one existing credit-card client.

The target measures whether the client defaulted in the following month.
The observed default rate was 22.12%.

No missing values were identified. There were 0 duplicate rows and
0 duplicated customer IDs.

The principal data-quality concerns were:
1. The education and marital-status variables contained undocumented category
   codes, including education codes 0, 5 and 6 and marital-status code 0.
2. The repayment-status variables contained -2 and 0 codes whose
   meanings were not fully documented. These will be investigated and handled
   explicitly during preprocessing.
3. Some monthly bill balances were negative. These were retained because they
   may represent account credits, refunds or overpayments rather than errors.

Because the data represent existing cardholders rather than all applicants,
the later decision-policy analysis will be presented as a hypothetical
risk-management simulation rather than a historical loan-approval evaluation.
This limitation is related to selection bias: existing cardholders may differ
systematically from the broader population because the original lender had 
already screened applicants.